In [1]:
import os
import re
import shutil
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
import unicodedata
import urllib3
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

In [2]:
# fra-eng.zip 다운 사이트 : https://www.manythings.org/anki/

In [3]:
path = os.getcwd()
with zipfile.ZipFile('fra-eng.zip', 'r') as zip_ref:
    zip_ref.extractall(path)

In [4]:
num_samples = 33000

In [5]:
def to_ascii(s):
    # 예시 : 'déjà diné' -> deja dine
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

In [6]:
def preprocess_sentence(sent):
    sent = to_ascii(sent.lower())
    sent = re.sub(r"([?.!,¿])", r" \1", sent)
    sent = re.sub(r"[^a-zA-Z!.?]+", r" ", sent)
    sent = re.sub(r"\s+", " ", sent)
    return sent

In [7]:
en_sent = u"Have you had dinner?"
fr_sent = u"Avez-vous déjà diné?"

print('전처리 전 영어 문장 :', en_sent)
print('전처리 후 영어 문장 :', preprocess_sentence(en_sent))
print('전처리 전 영어 문장 :', fr_sent)
print('전처리 후 영어 문장 :', preprocess_sentence(fr_sent))

전처리 전 영어 문장 : Have you had dinner?
전처리 후 영어 문장 : have you had dinner ?
전처리 전 영어 문장 : Avez-vous déjà diné?
전처리 후 영어 문장 : avez vous deja dine ?


In [8]:
def load_preprocessed_data():
    # 영어문장, 시작 토큰이 붙은 프랑스어 문장(디코더 입력), 종료 토큰이 붙은 프랑스어 문장(디코더 정답) 저장용 리스트
    encoder_input, decoder_input, decoder_target = [], [], []
    with open("fra.txt", "r", encoding='UTF8') as lines:
        for i, line in enumerate(lines):
            src_line, tar_line, _ = line.strip().split('\t')
            src_line = [w for w in preprocess_sentence(src_line).split()]

            tar_line = preprocess_sentence(tar_line) # 프랑스어 문장 전처리
            tar_line_in = [w for w in ("<sos> "+ tar_line).split()] # 프랑스어 문장 맨 앞에 시작 토큰 붙이고 공백 기준으로 쪼갬
            tar_line_out = [w for w in (tar_line + " <eos>").split()] # 전처리된 문장 맨 뒤에 종료 토큰 붙이고 공백 기준으로 쪼갬

            # 리스트 저장
            encoder_input.append(src_line)
            decoder_input.append(tar_line_in)
            decoder_target.append(tar_line_out)

            if i == num_samples -1:
                break

    return encoder_input, decoder_input, decoder_target

In [9]:
sents_en_in, sents_fra_in, sents_fra_out = load_preprocessed_data()
print('인코더의 입력 :',sents_en_in[:5])
print('디코더의 입력 :',sents_fra_in[:5])
print('디코더의 레이블 :',sents_fra_out[:5])

인코더의 입력 : [['go', '.'], ['go', '.'], ['go', '.'], ['go', '.'], ['hi', '.']]
디코더의 입력 : [['<sos>', 'va', '!'], ['<sos>', 'marche', '.'], ['<sos>', 'en', 'route', '!'], ['<sos>', 'bouge', '!'], ['<sos>', 'salut', '!']]
디코더의 레이블 : [['va', '!', '<eos>'], ['marche', '.', '<eos>'], ['en', 'route', '!', '<eos>'], ['bouge', '!', '<eos>'], ['salut', '!', '<eos>']]


In [10]:
tokenizer_en = Tokenizer(filters="", lower=False) # 영어 단어들을 숫자로 매핑하는 객체
tokenizer_en.fit_on_texts(sents_en_in) # 영어 단어들 인덱스 부여
encoder_input = tokenizer_en.texts_to_sequences(sents_en_in) # 텍스트 문장을 방금 만든 단어장을 바탕으로 정수 시퀀스(숫자 배열)로 변환
encoder_input = pad_sequences(encoder_input, padding="post") # 문장의 길이를 가장 긴 문장의 길이에 맞춰 통일

In [11]:
tokenizer_fra = Tokenizer(filters="", lower=False) # 프랑스어 단어들을 숫자로 매핑하는 객체

# 프랑스어 단어들 인덱스 부여
tokenizer_fra.fit_on_texts(sents_fra_in)
tokenizer_fra.fit_on_texts(sents_fra_out)

# 텍스트 문장을 정수 시퀀스(숫자 배열)로 변환 및 패딩
decoder_input = tokenizer_fra.texts_to_sequences(sents_fra_in)
decoder_input = pad_sequences(decoder_input, padding="post")

decoder_target =  tokenizer_fra.texts_to_sequences(sents_fra_out)
decoder_target = pad_sequences(decoder_target, padding="post")

In [12]:
src_vocab_size = len(tokenizer_en.word_index) + 1
tar_vocab_size = len(tokenizer_fra.word_index) + 1
print("단어 집합의 크기 : {:d}, 프랑스어 단어 집합의 크기 : {:d}".format(src_vocab_size, tar_vocab_size))

단어 집합의 크기 : 4519, 프랑스어 단어 집합의 크기 : 7912


In [13]:
print('인코더의 입력의 크기(shape) :',encoder_input.shape)
print('디코더의 입력의 크기(shape) :',decoder_input.shape)
print('디코더의 레이블의 크기(shape) :',decoder_target.shape)

인코더의 입력의 크기(shape) : (33000, 7)
디코더의 입력의 크기(shape) : (33000, 16)
디코더의 레이블의 크기(shape) : (33000, 16)


In [14]:
# 단어 사전 구축
src_to_index = tokenizer_en.word_index # 영어 단어를 키, 정수를 값으로 가짐
index_to_src = tokenizer_en.index_word # 정수를 키, 영어 단어를 값으로 가짐
tar_to_index = tokenizer_fra.word_index
index_to_tar = tokenizer_fra.index_word

In [15]:
indices = np.arange(encoder_input.shape[0])
np.random.shuffle(indices)
print("랜덤 시퀀스 :",indices)

랜덤 시퀀스 : [14223 30798  5733 ... 23768  1468 13359]


In [16]:
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]

In [17]:
encoder_input[30000]

array([ 3, 10, 36, 68,  1,  0,  0])

In [18]:
n_of_val = int(33000*0.1)
print('검증 데이터의 개수 :',n_of_val)

검증 데이터의 개수 : 3300


In [20]:
encoder_input_train = encoder_input[:-n_of_val]
decoder_input_train = decoder_input[:-n_of_val]
decoder_target_train = decoder_target[:-n_of_val]

encoder_input_test = encoder_input[-n_of_val:]
decoder_input_test = decoder_input[-n_of_val:]
decoder_target_test = decoder_target[-n_of_val:]

In [21]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Masking
from tensorflow.keras.models import Model

embedding_dim = 64
hidden_units = 64

In [22]:
encoder_inputs = Input(shape=(None,)) # 인코더 입력
enc_emb = Embedding(src_vocab_size, embedding_dim)(encoder_inputs) # 인코더 임베딩
enc_masking = Masking(mask_value=0.0)(enc_emb) # 마스킹 -> 패당 과정에서 채운 0을 스킵함
encoder_lstm = LSTM(hidden_units, return_state=True) # 은닉층, return_state=True -> 출력값뿐만 아니라 은닉상태, 셀 상태도 출력하게 함
encoder_outputs, state_h, state_c = encoder_lstm(enc_masking) # 인코더 출력, 은닉층(단기 기억 벡터) 출력, 셀상태(장기 기억) 출력
encoder_states = [state_h, state_c]

In [23]:
decoder_inputs = Input(shape=(None,)) # 디코더 입력
dec_emb_layer = Embedding(tar_vocab_size, hidden_units) # 디코더 임베딩
dec_emb = dec_emb_layer(decoder_inputs) 
dec_masking = Masking(mask_value=0.0)(dec_emb) # 마스크
decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True) # 은닉층 설계
decoder_outputs, _, _ = decoder_lstm(dec_masking, initial_state=encoder_states)

decoder_dense = Dense(tar_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [25]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['acc'])
model.fit(x=[encoder_input_train, decoder_input_train], y=decoder_target_train,
          validation_data = ([encoder_input_test, decoder_input_test], decoder_target_test),
          batch_size=128, epochs=50)

Epoch 1/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 25s 93ms/step - acc: 0.6187 - loss: 3.3102 - val_acc: 0.6208 - val_loss: 2.0462
Epoch 2/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 22s 94ms/step - acc: 0.7010 - loss: 1.8632 - val_acc: 0.7318 - val_loss: 1.7499
Epoch 3/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 22s 95ms/step - acc: 0.7413 - loss: 1.6505 - val_acc: 0.7540 - val_loss: 1.5744
Epoch 4/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 27s 117ms/step - acc: 0.7629 - loss: 1.4969 - val_acc: 0.7674 - val_loss: 1.4563
Epoch 5/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 27s 115ms/step - acc: 0.7753 - loss: 1.3931 - val_acc: 0.7804 - val_loss: 1.3696
Epoch 6/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 29s 123ms/step - acc: 0.7891 - loss: 1.3033 - val_acc: 0.7976 - val_loss: 1.2909
Epoch 7/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 30s 129ms/step - acc: 0.8025 - loss: 1.2269 - val_acc: 0.8040 - val_loss: 1.2304
Epoch 8/50
233/233 ━━━━━━━━━━━━━━━━━━━━ 26s 111ms/step - acc: 0.8101 - loss: 1.1634 - val_acc: 0.8130 - val_loss: 1.1786
Epoch 9/50
233/233 ━━━━━━━━━━━━━━━━

In [26]:
encoder_model = Model(encoder_inputs, encoder_states) # 인코더

# 디코더
decoder_state_input_h = Input(shape=(hidden_units,))
decoder_state_input_c = Input(shape=(hidden_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# 임베딩 및 LSTM 연결
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs2] + decoder_states2)

In [35]:
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1,1))
    target_seq[0,0] = tar_to_index['<sos>']

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = index_to_tar[sampled_token_index]
        decoded_sentence += ' '+sampled_char

        if (sampled_char == '<eos>' or len(decoded_sentence) > 50):
            stop_condition = True

        target_seq = np.zeros((1,1))
        target_seq[0,0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence

In [36]:
def seq_to_src(input_seq):
    sentence = ''
    for encoded_word in input_seq:
        if(encoded_word != 0):
            sentence = sentence + index_to_src[encoded_word] + ' '
    return sentence

In [37]:
def seq_to_tar(input_seq):
    sentence = ''
    for encoded_word in input_seq:
        if(encoded_word != 0 and encoded_word != tar_to_index['<sos>'] and encoded_word != tar_to_index['<eos>']):
            sentence = sentence + index_to_tar[encoded_word] + ' '
    return sentence

In [38]:
for seq_index in [3, 50, 100, 300, 1001]:
    input_seq = encoder_input_train[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("입력문장 :", seq_to_src(encoder_input_train[seq_index]))
    print("정답문장 :", seq_to_tar(decoder_input_train[seq_index]))
    print("번역문장 :", decoded_sentence[1:-5])
    print("-"*50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
입력문장 : i took a shortcut . 
정답문장 : j ai pris un raccourci . 
번역문장 : j ai pris un raccourci . 
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
입력문장 : do you think so ? 
정답문장 : penses tu la meme chose ? 
번역문장 : est ce que tu y aller comment ? 
--------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━